# DataPatch walkthrough

The web app is the main way to use DataPatch, but the cleaning engine is a plain Python package.
This notebook runs the same steps as the live demo: load the messy sample, look at the profile,
clean a few columns and compare the quality scores before and after.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Make the package importable when the notebook is opened from the notebooks/ folder.
ROOT = Path.cwd() if (Path.cwd() / "datapatch").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from datapatch import Workspace, profile_frame  # noqa: E402
from datapatch.files import read_csv  # noqa: E402

raw = read_csv((ROOT / "sample_data" / "messy_customers.csv").read_bytes())
raw.head(10)

## Profile the original data

In [ ]:
report = profile_frame(raw)
report["summary"]

In [ ]:
issues = [
    {
        "column": col["name"],
        "type": col["kind"],
        "issue": issue["message"],
        "suggested fix": (issue["fix"] or {}).get("label"),
    }
    for col in report["columns"]
    for issue in col["issues"]
]
pd.DataFrame(issues)

## Clean it step by step

A `Workspace` keeps the original frame and a list of steps, so anything can be undone.

In [ ]:
ws = Workspace(raw, "messy_customers.csv")

steps = [
    ("remove_duplicates", {}),
    ("trim_whitespace", {}),
    ("mark_missing", {}),
    ("standardize_categories", {"column": "city"}),
    ("standardize_categories", {"column": "plan"}),
    ("change_case", {"columns": ["full_name"], "mode": "title"}),
    ("convert_type", {"columns": ["signup_date"], "target": "date"}),
    ("convert_type", {"columns": ["age"], "target": "integer"}),
    ("convert_type", {"columns": ["annual_spend"], "target": "number"}),
    ("convert_type", {"columns": ["active"], "target": "boolean"}),
    ("handle_outliers", {"column": "age", "method": "missing"}),
    ("fill_missing", {"columns": ["age", "annual_spend"], "strategy": "median"}),
]
for op, params in steps:
    step = ws.apply(op, params)
    print(f"{step.label:<55} score {ws.current_profile['summary']['score']}")

In [ ]:
# Undo the last step, then bring it back.
print("undo:", ws.undo().label)
print("redo:", ws.redo().label)

## Compare original and cleaned

In [ ]:
before = ws.original_profile["summary"]
after = ws.current_profile["summary"]
keys = ["score", "rows", "missing_cells", "duplicate_rows", "inconsistent_cells", "type_issues"]
pd.DataFrame({"original": [before[k] for k in keys], "cleaned": [after[k] for k in keys]}, index=keys)

In [ ]:
pd.DataFrame({"original": before["components"], "cleaned": after["components"]})

In [ ]:
ws.current.dtypes

## What changed, and a reusable recipe

In [ ]:
changes = ws.changes
print(f"{changes['cells_changed']} cells changed, {changes['rows_removed']} rows removed")
pd.Series(changes["by_column"], name="cells changed")

In [ ]:
from datapatch import apply_recipe, make_recipe

recipe = make_recipe(ws.steps, source=ws.filename)
# The same recipe cleans another export of this data in one call.
again, report = apply_recipe(raw, recipe)
print(report.summary())
print("same result as the session:", again.equals(ws.current))

In [ ]:
# Save the result next to the sample (ignored by git).
out = ROOT / "sample_data" / "messy_customers_cleaned.csv"
ws.current.to_csv(out, index=False)
print("saved", out.name, ws.current.shape)